In [1]:
import os
import json
import pandas as pd
import re

In [3]:
samples_path = "../Definition-generation-for-LSCD/generated_definitions/english/greedy" # git clone git@github.com:ltgoslo/Definition-generation-for-LSCD.git
time_period_1_path = os.path.join(samples_path, "english-corpus1.tsv.gz")
time_period_1 = pd.read_csv(time_period_1_path, sep='\t', compression="gzip", header=None)
print(time_period_1.head())


      0                                                  1  \
0  part  Most of the asp * ing men of that part of Alab...   
1  face  The faces are umajured , except that the color...   
2   tip  The moon was in its western field , white and ...   
3  tree  The well known fact that we may have a hundred...   
4  face  Rising upon the instant in his stirrups , the ...   

                                                   2  
0              A portion ; some portion ; a section.  
1                   The front surface of a painting.  
2                  To cover the top of ; to overlay.  
3  A plant that bears a given fruit or seed ; spe...  
4  The front of the skull, especially the front o...  


In [7]:
time_period_1[1].iloc[2]

'The moon was in its western field , white and clear as silver , with which it tipped the bursting waves . What is the definition of tip?'

In [10]:
with open("english-corpus1.jsonl", "w") as f:
    for i, record in time_period_1.iterrows():
        line = json.dumps(
            {
                'target': record[0],
                'example': record[1].split("What is the definition of ")[0].strip(),
            }
        ) + '\n'
        f.write(line)
        if i < 5:
            print(line)

{"target": "part", "example": "Most of the asp * ing men of that part of Alabama fell in with Mr. Lewis opinions ; but Mr."}

{"target": "face", "example": "The faces are umajured , except that the colors are not so bright as at a former period ."}

{"target": "tip", "example": "The moon was in its western field , white and clear as silver , with which it tipped the bursting waves ."}

{"target": "tree", "example": "The well known fact that we may have a hundred different varieties of pear upon the same tree , each of which produces it s fruit of the proper form , colour , and quality ; and that we may have , at least for a time , several distinct , though nearly related species upon one stock , as the Peach , Apricot , Nectarine , and Plum , prove"}

{"target": "face", "example": "Rising upon the instant in his stirrups , the bold cavalier hurled , with a sure and steady hand , the discharged weapon in the face of his next opponent ."}



In [11]:
time_period_2_path = os.path.join(samples_path, "english-corpus2.tsv.gz")
time_period_2 = pd.read_csv(time_period_2_path, sep='\t', compression="gzip", header=None)
with open("english-corpus2.jsonl", "w") as f:
    for i, record in time_period_2.iterrows():
        line = json.dumps(
            {
                'target': record[0],
                'example': record[1].split("What is the definition of ")[0].strip(),
            }
        ) + '\n'
        f.write(line)
        if i < 5:
            print(line)

{"target": "record", "example": "Even as it is , with infusions of Wendish and Polish and German and Swedish blood , we come close to being a pure strain , and unlike primitive endogamous groups , we have kept records ."}

{"target": "stroke", "example": "Lou moved his hand under the covers , stroked Karen 's thigh , her hip , the smooth round point of her pelvis under the nightie ."}

{"target": "circle", "example": "Big as the Soviet bomb was , it s waves were far weaker than the volcano 's , but the time they took to circle the earth was almost exactly the same : 36 hr."}

{"target": "word", "example": "I think there 's something very direct and refreshing about them , and something very good about the words of their songs , \" she said . \""}

{"target": "word", "example": "Jenny bobs her head and mouths some words to make fun of Sally , then turns to you : \""}



In [12]:
time_period_2.shape

(30785, 3)

In [13]:
time_period_1.shape

(26844, 3)

In [5]:
SYSTEM_PATTERN = re.compile(
    r"""<\|begin_of_text\|><\|start_header_id\|>system<\|end_header_id\|>\n\nYou are a lexicographer familiar with providing concise definitions of word meanings.<\|eot_id\|><\|start_header_id\|>user<\|end_header_id\|>\n\n"""
)
TARGET_PATTERN = re.compile(
        r'Please provide a concise definition for the meaning of the word \"(.+)\" in the following sentence',
    )
TO_SPLIT = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


In [12]:
our = "english-corpus1.jsonl.cluster-work-projects-nn9851k-mariiaf-periti_2025_data-dpo_model-lora-True-lr-1e-06-lora-16-bf16-linear-beta-0.5-warmup-0.txt"

outdir = "../Definition-generation-for-LSCD/generated_definitions/english/dpo"
os.makedirs(outdir, exist_ok=True)

In [13]:
def convert_to_2024_format(our, period):
    to_replace = f"corpus{period}"
    our = our.replace("corpus1", to_replace)
    with open(our, "r" ) as f:
        generated = f.read()
    splitted = re.split(SYSTEM_PATTERN, generated)  # not really memory efficient, but we are safe of \n inside examples and such
    assert len(splitted) > 1
    to_df = [[], [], []]
    for text in splitted:
        text = text.strip()
        if text:
            prompt, definition = text.split(TO_SPLIT)
            prompt_with_target, example = prompt.split(": ", maxsplit=1)
            target = re.match(TARGET_PATTERN, prompt_with_target).group(1)
            to_df[0].append(target)
            to_df[1].append(example)
            to_df[2].append(definition)
    our_corpus = pd.DataFrame(to_df).T
    print(our_corpus.head())
    our_corpus.to_csv(
        os.path.join(outdir, f"english-corpus{period}.tsv.gz"), sep='\t', compression="gzip", index=None,
    )

In [14]:
convert_to_2024_format(our, 1)

      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  word  immaturity , and want of experience -- besides...   
4  part  immaturity , and want of experience -- besides...   

                                                   2  
0            The topmost, foremost, or leading part.  
1  (Figurative) The most important part or one of...  
2  The smallest discrete unit of written language...  
3                     (Obsolete) A proverb or motto.  
4                  A fragment of something; a piece.  


In [15]:
convert_to_2024_format(our, 2)

        0                                                  1  \
0    risk  and executives -- Epidemiologists -- Community...   
1    head  , but before he can see who this is , or make ...   
2  record  60 Number of home runs Mark McGwire has hit 58...   
3  player  60 Number of home runs Mark McGwire has hit 58...   
4    face  after , one DOES realize , I just did wish I '...   

                                                   2  
0  (Uncountable) The condition of being vulnerabl...  
1  (Countable) The part of the body of an animal ...  
2  The most extreme known value of a variable, ei...  
3  (Baseball) A player on one or more teams in a ...  
4  (Anatomy) The front part of the head of a huma...  


In [17]:
orig = "english-corpus1.jsonl.cluster-work-projects-nn9851k-mariiaf-periti_2025_data-LlamaDictionary-en_ML38BI-merged.txt"

outdir = "../Definition-generation-for-LSCD/generated_definitions/english/periti_2025"
os.makedirs(outdir, exist_ok=True)

In [18]:
convert_to_2024_format(orig, 1)
convert_to_2024_format(orig, 2)

      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  word  immaturity , and want of experience -- besides...   
4  part  immaturity , and want of experience -- besides...   

                                                   2  
0                    (Figurative) A leader or chief.  
1  (Figurative) The most important part or piece ...  
2  The smallest discrete unit of written language...  
3    (Obsolete) Something spoken; speech, discourse.  
4  (Obsolete) A third; a third part; a share, a p...  
        0                                                  1  \
0    risk  and executives -- Epidemiologists -- Community...   
1    head  , but before he can see who this is , or make ...   
2  player  60 Number of home runs Mark McGwire has hit 58...   
3  record  60 Number of home runs Mark Mc

In [19]:
old = "english-corpus1.jsonl.FrancescoPeriti-Llama3Dictionary-merge.txt"
outdir = "../Definition-generation-for-LSCD/generated_definitions/english/periti_2024"
os.makedirs(outdir, exist_ok=True)
convert_to_2024_format(old, 1)
convert_to_2024_format(old, 2)

      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  part  immaturity , and want of experience -- besides...   
4  word  immaturity , and want of experience -- besides...   

                                                   2  
0            A superior or governing body or member.  
1  The top of the head of a person, or the upper ...  
2  A unit of language that native speakers can id...  
3                             A fraction of a whole.  
4  A single distinct conceptual unit of language,...  
        0                                                  1  \
0    risk  and executives -- Epidemiologists -- Community...   
1    head  , but before he can see who this is , or make ...   
2  record  60 Number of home runs Mark McGwire has hit 58...   
3  player  60 Number of home runs Mark Mc